# Brain Tumor MRI Classification: ResNet50, GWN and LIME

Experiments accompanying *Beyond Accuracy: Comparison of ResNet50 and GWN-Enhanced Models for Brain Tumor MRI Classification with LIME Visualization* (ICTIS 2026).

Compare two ResNet50 configurations on three classes, inspect training dynamics, and explain the same five test images with LIME. This notebook contains no recorded training outputs; paper numbers are not asserted as reproduced results.

## 1. Configuration
The paper specifies 1,100 images per class, an 80/10/10 split, RGB 224×224 inputs, ImageNet normalization, horizontal flipping and ±15° rotation. Training uses dropout 0.5, batch size 32, Adam, five head-only warm-up epochs at 1e-4, then at most 30 fine-tuning epochs at 5e-5.

Edit `DATA_ROOTS` before running. The default pools Kaggle Training and Testing folders before making a new split, following section 2.2. Set `(Path('./Training'),)` to retain the source notebook's training-folder-only pool; these settings do not produce the same split.

Seed 42, weight decay 1e-4, scheduler patience/factor, early-stopping patience, GWN group count and LIME sampling parameters are source-code choices; the paper does not specify all of them. Do not infer an exact numerical reproduction from matching the broad configuration.

In [ ]:
# Uncomment if the packages are not installed
# !pip install -q lime scikit-image

import random
import copy
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.preprocessing import label_binarize

from lime import lime_image
from skimage.segmentation import mark_boundaries
from tqdm.auto import tqdm


DATA_ROOTS = (Path("./Training"), Path("./Testing"))
OUTPUT_ROOT = Path("./results")
from datetime import datetime, timezone
import json
RUN_DIR = OUTPUT_ROOT / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_DIR.mkdir(parents=True, exist_ok=False)

SEED = 42
N_PER_CLASS = 1100
IMG_SIZE = 224
BATCH_SIZE = 32

PHASE1_EPOCHS = 5
PHASE2_EPOCHS = 30
PATIENCE = 3

LR_PHASE1 = 1e-4
LR_PHASE2 = 5e-5
WEIGHT_DECAY = 1e-4

GWN_GROUPS = 2
DROPOUT = 0.5
LIME_IMAGES = 5
LIME_SAMPLES = 1000
LIME_FEATURES = 5
NUM_WORKERS = 0  # portable default for notebooks on Windows and Colab

MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 2. Dataset selection and split (paper Table 1)
Class order: 0 = no tumor, 1 = glioma, 2 = pituitary. Meningioma is excluded.
A seeded sample of 1,100 images per class is stratified into 880 train, 110 validation and 110 test images per class.
The saved manifests identify the actual selected files. Disjoint paths do not establish patient independence or exclude near-duplicate images; patient IDs are unavailable in this workflow.

In [ ]:
# Expected folders: glioma, notumor, pituitary
CLASS_NAMES = ["notumor", "glioma", "pituitary"]
extensions = {".jpg", ".jpeg", ".png", ".bmp"}

rows = []
for label, class_name in enumerate(CLASS_NAMES):
    paths = []
    for root in DATA_ROOTS:
        class_dir = root / class_name
        if not class_dir.is_dir():
            raise FileNotFoundError(f"Missing class directory: {class_dir}. Check DATA_ROOTS.")
        paths.extend(p.resolve() for p in class_dir.iterdir() if p.is_file() and p.suffix.lower() in extensions)
    paths = sorted(set(paths))

    if len(paths) < N_PER_CLASS:
        raise ValueError(f"{class_name}: found {len(paths)} images, need {N_PER_CLASS}")

    rng = random.Random(SEED)
    selected = rng.sample(paths, N_PER_CLASS)

    rows.extend(
        {"path": str(path), "label": label, "class_name": class_name}
        for path in selected
    )

df = pd.DataFrame(rows)

train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df["label"],
    random_state=SEED
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=SEED
)

for name, split in [("train", train_df), ("val", val_df), ("test", test_df)]:
    counts = split["class_name"].value_counts().reindex(CLASS_NAMES)
    print(name, len(split), counts.to_dict())

assert len(train_df) == 2640 and len(val_df) == 330 and len(test_df) == 330
assert set(train_df.path).isdisjoint(val_df.path)
assert set(train_df.path).isdisjoint(test_df.path)
assert set(val_df.path).isdisjoint(test_df.path)
for name, split in [("train", train_df), ("val", val_df), ("test", test_df)]:
    split.to_csv(RUN_DIR / f"{name}_manifest.csv", index=False)
config = {key: value for key, value in list(globals().items())
          if key.isupper() and isinstance(value, (int, float, str, list))}
config.update(data_roots=[str(p) for p in DATA_ROOTS], torch_version=torch.__version__)
(RUN_DIR / "config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")


## 3. Preprocessing and loaders (section 2.3)
Resize, tensor conversion and ImageNet normalization apply to every split. Random flipping and rotation apply only to training; validation/test preserve a fixed order for matching predictions with filenames.

In [ ]:
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class MRIDataset(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        image = Image.open(row["path"]).convert("RGB")
        return self.transform(image), int(row["label"])

train_ds = MRIDataset(train_df, train_tf)
val_ds = MRIDataset(val_df, eval_tf)
test_ds = MRIDataset(test_df, eval_tf)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available()
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available()
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available()
)


## 4. ResNet50 and supplied GWN implementation
Both models use ImageNet weights and the same dropout/linear head. The GWN functions below use the normalization implemented in the original notebook.

**Implementation detail requiring confirmation:** normalization is over the last dimension after reshaping to `(groups, outputs_per_group, -1)`. This normalizes each output filter/vector independently; changing `ghost_groups` does not change that operation apart from padding and numerical round-off. The paper describes partitioned weight groups but equation (1) alone does not resolve the intended axes. An exact group-wise GWN claim needs confirmation against the original experiment implementation (paper reference 11).

In [ ]:
class GWNConv2d(nn.Conv2d):
    def __init__(self, *args, ghost_groups=2, eps=1e-6, **kwargs):
        super().__init__(*args, **kwargs)
        self.ghost_groups = max(1, ghost_groups)
        self.eps = eps

    def forward(self, x):
        w = self.weight
        out_channels = w.shape[0]
        groups = min(self.ghost_groups, out_channels)

        pad = (-out_channels) % groups
        if pad:
            w = F.pad(w, (0, 0, 0, 0, 0, 0, 0, pad))

        shape = w.shape
        w = w.reshape(groups, shape[0] // groups, -1)
        w = w / torch.sqrt((w ** 2).sum(dim=-1, keepdim=True) + self.eps)
        w = w.reshape(shape)[:out_channels]

        return F.conv2d(
            x, w, self.bias, self.stride,
            self.padding, self.dilation, self.groups
        )


class GWNLinear(nn.Linear):
    def __init__(self, *args, ghost_groups=2, eps=1e-6, **kwargs):
        super().__init__(*args, **kwargs)
        self.ghost_groups = max(1, ghost_groups)
        self.eps = eps

    def forward(self, x):
        w = self.weight
        out_features = w.shape[0]
        groups = min(self.ghost_groups, out_features)

        pad = (-out_features) % groups
        if pad:
            w = F.pad(w, (0, 0, 0, pad))

        shape = w.shape
        w = w.reshape(groups, shape[0] // groups, -1)
        w = w / torch.sqrt((w ** 2).sum(dim=-1, keepdim=True) + self.eps)
        w = w.reshape(shape)[:out_features]

        return F.linear(x, w, self.bias)


def apply_gwn(module, ghost_groups=2):
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Conv2d):
            new = GWNConv2d(
                child.in_channels,
                child.out_channels,
                child.kernel_size,
                stride=child.stride,
                padding=child.padding,
                dilation=child.dilation,
                groups=child.groups,
                bias=child.bias is not None,
                padding_mode=child.padding_mode,
                ghost_groups=ghost_groups,
            )
            new.load_state_dict(child.state_dict())
            setattr(module, name, new)

        elif isinstance(child, nn.Linear):
            new = GWNLinear(
                child.in_features,
                child.out_features,
                bias=child.bias is not None,
                ghost_groups=ghost_groups,
            )
            new.load_state_dict(child.state_dict())
            setattr(module, name, new)

        else:
            apply_gwn(child, ghost_groups)

    return module


In [ ]:
def build_model(use_gwn=False):
    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    in_features = model.fc.in_features

    model.fc = nn.Sequential(
        nn.Dropout(DROPOUT),
        nn.Linear(in_features, len(CLASS_NAMES))
    )

    if use_gwn:
        model = apply_gwn(model, GWN_GROUPS)

    return model


## 5. Shared two-phase training (section 2.4)
The same function trains both models. Best weights are selected using validation loss during fine-tuning, as in the supplied notebook; no test metric selects the checkpoint. The warm-up freezes parameter gradients, but `model.train()` still updates BatchNorm running statistics. This source behavior is retained and should not be described as freezing all backbone state.

Histories record phase, within-phase epoch and total epoch. A stable loss plateau (approximately 4–5 versus 7–8 epochs in the paper) is distinct from an optimal stopping/checkpoint epoch (approximately 10 versus 23). These are reported observations, not hardcoded training targets.

In [ ]:
criterion = nn.CrossEntropyLoss()

def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    total = 0

    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        for images, labels in tqdm(loader, leave=False):
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            if is_train:
                optimizer.zero_grad()

            logits = model(images)
            loss = criterion(logits, labels)

            if is_train:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * labels.size(0)
            total += labels.size(0)

    return total_loss / total


In [ ]:
def train_model(name, use_gwn=False):
    model = build_model(use_gwn=use_gwn).to(DEVICE)

    # warm-up: train only the classifier
    for p in model.parameters():
        p.requires_grad = False
    for p in model.fc.parameters():
        p.requires_grad = True

    optimizer = torch.optim.Adam(
        model.fc.parameters(),
        lr=LR_PHASE1,
        weight_decay=WEIGHT_DECAY
    )

    history = {"train_loss": [], "val_loss": []}

    for epoch in range(PHASE1_EPOCHS):
        train_loss = run_epoch(model, train_loader, optimizer)
        val_loss = run_epoch(model, val_loader)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        print(
            f"warm-up {epoch + 1}/{PHASE1_EPOCHS} | "
            f"train {train_loss:.4f} | val {val_loss:.4f}"
        )

    # fine-tune all layers
    for p in model.parameters():
        p.requires_grad = True

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LR_PHASE2,
        weight_decay=WEIGHT_DECAY
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=2
    )

    best_val = float("inf")
    best_state = None
    wait = 0
    best_epoch = 0

    for epoch in range(PHASE2_EPOCHS):
        train_loss = run_epoch(model, train_loader, optimizer)
        val_loss = run_epoch(model, val_loader)
        scheduler.step(val_loss)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        print(
            f"fine-tune {epoch + 1}/{PHASE2_EPOCHS} | "
            f"train {train_loss:.4f} | val {val_loss:.4f}"
        )

        if val_loss < best_val:
            best_val = val_loss
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1
            wait = 0
        else:
            wait += 1

        if wait >= PATIENCE:
            print("early stopping")
            break

    if best_state is None:
        raise RuntimeError("No finite validation checkpoint was produced")
    model.load_state_dict(best_state)
    torch.save(best_state, RUN_DIR / f"{name}_best.pth")
    pd.DataFrame({"phase": ["warmup"] * PHASE1_EPOCHS + ["finetune"] * (len(history["train_loss"])-PHASE1_EPOCHS),
                  "total_epoch": range(1, len(history["train_loss"])+1),
                  "train_loss": history["train_loss"], "val_loss": history["val_loss"]}).to_csv(RUN_DIR / f"{name}_history.csv", index=False)
    return model, history, {"best_finetune_epoch": best_epoch, "best_total_epoch": PHASE1_EPOCHS + best_epoch, "best_val_loss": best_val}


In [ ]:
baseline, hist_baseline, selection_baseline = train_model("resnet50", use_gwn=False)
gwn_model, hist_gwn, selection_gwn = train_model("resnet50_gwn", use_gwn=True)
selection = {"ResNet50": selection_baseline, "ResNet50+GWN": selection_gwn}
(RUN_DIR / "checkpoint_selection.json").write_text(json.dumps(selection, indent=2))
display(pd.DataFrame(selection).T)


## 6. Evaluation and training curves (section 2.5)
Metrics below are computed from the selected checkpoints on the held-out test images, never filled from paper tables.
Sensitivity is recall; specificity uses a one-vs-rest confusion matrix. AUC uses probabilities. Per-class one-vs-rest accuracy is explicitly named to distinguish it from overall multiclass accuracy.

The paper reports average accuracy 99.9%. A single 330-image test evaluation changes in increments of 1/330 ≈ 0.303 percentage points; the supplied notebook contains no run outputs or averaging protocol establishing 99.9%. The exact denominator/aggregation requires original logs.

In [ ]:
def predict(model, loader):
    model.eval()
    y_true, y_pred, y_prob = [], [], []

    with torch.no_grad():
        for images, labels in loader:
            logits = model(images.to(DEVICE))
            prob = F.softmax(logits, dim=1).cpu().numpy()

            y_true.extend(labels.numpy())
            y_pred.extend(prob.argmax(axis=1))
            y_prob.extend(prob)

    return (
        np.asarray(y_true),
        np.asarray(y_pred),
        np.asarray(y_prob)
    )


def evaluate(model, name):
    y_true, y_pred, y_prob = predict(model, test_loader)
    y_bin = label_binarize(y_true, classes=np.arange(len(CLASS_NAMES)))
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(CLASS_NAMES)))

    sensitivity = np.diag(cm) / cm.sum(axis=1)

    specificity = []
    for i in range(len(CLASS_NAMES)):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = cm.sum() - tp - fn - fp
        specificity.append(tn / (tn + fp))

    per_class_auc = roc_auc_score(
        y_bin, y_prob, average=None, multi_class="ovr"
    )

    summary = {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_auc": roc_auc_score(
            y_bin, y_prob, average="macro", multi_class="ovr"
        ),
        "macro_f1": f1_score(y_true, y_pred, average="macro")
    }

    per_class = pd.DataFrame({
        "class": CLASS_NAMES,
        "AUC": per_class_auc,
        "sensitivity": sensitivity,
        "specificity": specificity
    })

    per_class["f1"] = f1_score(y_true, y_pred, labels=np.arange(len(CLASS_NAMES)), average=None, zero_division=0)
    per_class["ovr_accuracy"] = [np.mean((y_true == i) == (y_pred == i)) for i in range(len(CLASS_NAMES))]
    summary["sensitivity_macro"] = float(np.mean(sensitivity))
    summary["specificity_macro"] = float(np.mean(specificity))
    predictions = test_df.reset_index(drop=True)[["path", "label", "class_name"]].copy()
    predictions["prediction"] = y_pred
    for i, class_name in enumerate(CLASS_NAMES): predictions[f"p_{class_name}"] = y_prob[:, i]
    predictions.to_csv(RUN_DIR / f"{name}_predictions.csv", index=False)
    per_class.to_csv(RUN_DIR / f"{name}_per_class.csv", index=False)
    pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(RUN_DIR / f"{name}_confusion_matrix.csv")
    return summary, per_class, cm


base_summary, base_class, base_cm = evaluate(baseline, "resnet50")
gwn_summary, gwn_class, gwn_cm = evaluate(gwn_model, "resnet50_gwn")

display(pd.DataFrame(
    [base_summary, gwn_summary],
    index=["ResNet50", "ResNet50 + GWN"]
))

print("\nResNet50")
display(base_class)

print("\nResNet50 + GWN")
display(gwn_class)

pd.DataFrame([base_summary, gwn_summary], index=["ResNet50", "ResNet50+GWN"]).to_csv(RUN_DIR / "metrics.csv")


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(hist_baseline["train_loss"], label="ResNet50 train")
plt.plot(hist_baseline["val_loss"], label="ResNet50 val")
plt.plot(hist_gwn["train_loss"], label="GWN train")
plt.plot(hist_gwn["val_loss"], label="GWN val")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.legend()
plt.tight_layout()
plt.savefig(RUN_DIR / "training_curves.png", dpi=160)
plt.show()


## 7. LIME comparison on the same five test images (section 2.6 / Figure 3)
Keep the LIME section: it is central to the paper's comparison. Both models explain the same seeded sample of five images, using positively contributing superpixels for each model's predicted class. Different predicted classes mean different explanation targets.

Each model/image call gets a fresh explainer with the same seed, avoiding sequential random-state drift between models. `num_samples=1000` and five visible regions come from the supplied notebook, not a fully specified protocol in the paper. Selected filenames, masks and figures are saved. LIME regions are perturbation sensitivities, not tumor segmentations; image-level splitting and background reliance remain limitations.

In [ ]:
lime_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

def lime_predict(model, images):
    model.eval()
    batch = []
    for image in images:
        image = np.clip(image, 0, 1)
        pil = Image.fromarray((image * 255).astype(np.uint8))
        batch.append(lime_tf(pil))

    batch = torch.stack(batch).to(DEVICE)

    with torch.no_grad():
        return F.softmax(model(batch), dim=1).cpu().numpy()


def load_lime_image(idx):
    row = test_df.reset_index(drop=True).iloc[idx]
    image = Image.open(row["path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    return np.asarray(image, dtype=np.float32) / 255.0, int(row["label"])


rng = np.random.default_rng(SEED)
sample_indices = rng.choice(len(test_df), size=LIME_IMAGES, replace=False)

lime_dir = RUN_DIR / "lime"
lime_dir.mkdir()
test_df.reset_index(drop=True).iloc[sample_indices].to_csv(lime_dir / "selected_images.csv", index=False)
for idx in sample_indices:
    image, true_label = load_lime_image(idx)

    exp_base = lime_image.LimeImageExplainer(random_state=SEED).explain_instance(
        image,
        classifier_fn=lambda x: lime_predict(baseline, x),
        top_labels=1,
        num_samples=LIME_SAMPLES,
        random_seed=SEED
    )

    exp_gwn = lime_image.LimeImageExplainer(random_state=SEED).explain_instance(
        image,
        classifier_fn=lambda x: lime_predict(gwn_model, x),
        top_labels=1,
        num_samples=LIME_SAMPLES,
        random_seed=SEED
    )

    pred_base = exp_base.top_labels[0]
    pred_gwn = exp_gwn.top_labels[0]

    vis_base, mask_base = exp_base.get_image_and_mask(
        pred_base,
        positive_only=True,
        num_features=LIME_FEATURES,
        hide_rest=True
    )

    vis_gwn, mask_gwn = exp_gwn.get_image_and_mask(
        pred_gwn,
        positive_only=True,
        num_features=LIME_FEATURES,
        hide_rest=True
    )

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    axes[0].imshow(image)
    axes[0].set_title(f"True: {CLASS_NAMES[true_label]}")

    axes[1].imshow(mark_boundaries(vis_base, mask_base))
    axes[1].set_title(f"ResNet50: {CLASS_NAMES[pred_base]}")

    axes[2].imshow(mark_boundaries(vis_gwn, mask_gwn))
    axes[2].set_title(f"ResNet50 + GWN: {CLASS_NAMES[pred_gwn]}")

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    fig.savefig(lime_dir / f"test_{idx}_comparison.png", dpi=180)
    np.savez_compressed(lime_dir / f"test_{idx}_masks.npz", baseline=mask_base, gwn=mask_gwn)
    plt.show()
    plt.close(fig)


## 8. Interpretation and reproducibility limits
- The supplied notebook has no saved outputs or checkpoints; the figures and numbers in the manuscript have not been reproduced here.
- Pooling Training/Testing follows the paper's stated custom sampling; use the source pool setting only when intentionally reproducing the supplied notebook. Actual file lists are essential for exact comparisons.
- This workflow retains image-level sampling. It neither implements patient-wise splitting nor proves leakage has been eliminated.
- The group axes in GWN require confirmation. Do not adjust the implementation merely to obtain the paper's metrics.
- LIME comparison can reveal background sensitivity but does not establish pathology localization or clinical reliability without independent validation.
- Use saved split manifests, probabilities, histories and configuration when reporting a new run. Keep paper-reported and newly measured values separate.

### Sources
Sources: the ICTIS 2026 manuscript and `brain-tumor-resnet50-gwn-lime-github.ipynb`.
GWN: Baihaqi, Shalsadilla and Argaputri, *Enhancing ResNet with ghost weight normalization for improved retina disease classification*, IC-ITECHS 2024, cited as reference 11 in the manuscript.
LIME: Ribeiro, Singh and Guestrin, *Why Should I Trust You? Explaining the Predictions of Any Classifier*, KDD 2016 (paper reference 20).
Dataset: Nickparvar, *Brain Tumor MRI Dataset*, Kaggle 2021 (paper reference 12).